# Capitolo 5 — Produzione e scorte multiperiodali (LP)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/modellazione-pl/blob/main/notebooks/lab05_produzione.ipynb)

Caso di studio: azienda che produce 3 componenti (prodotti 1, 2, 3) su un orizzonte
di 3 mesi con una risorsa condivisa (ore macchina).

Contenuto:
1. LP a costo minimo con servizio obbligatorio
2. Prezzi ombra della capacità e analisi dei duali
3. Sensitività: costo ottimo al variare della capacità

Il capitolo completo — modello, dati, risultati e analisi di sensitività — è [sul sito](https://fabiofurini.github.io/modellazione-pl/organizzazione/).

## Preparazione

La cella qui sotto installa `gurobipy` e scarica `stile.py`, la palette comune
degli script del corso. La licenza inclusa nel pacchetto pip è limitata a **2000
variabili e 2000 vincoli**: tutti i modelli del laboratorio sono piccoli e ci
stanno con largo margine; la si supera solo ingrandendo molto le istanze. In quel
caso si attiva la licenza accademica gratuita da [portal.gurobi.com](https://portal.gurobi.com).

In [ ]:
# Ambiente: il solver e lo stile grafico del laboratorio.
# In locale usa il python/stile.py del repository; su Colab installa e scarica quello che manca.
import importlib.util
import subprocess
import sys
import urllib.request
from pathlib import Path

if importlib.util.find_spec("gurobipy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gurobipy", "matplotlib", "pandas", "scipy"], check=True)

if importlib.util.find_spec("stile") is None:
    locale = next((p for p in (Path("../python/stile.py"), Path("python/stile.py"))
                   if p.exists()), None)
    if locale is not None:
        sys.path.insert(0, str(locale.parent.resolve()))     # notebook aperto nel repository
    else:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/fabiofurini/modellazione-pl/main/python/stile.py", "stile.py")   # Colab

In [ ]:
from fractions import Fraction

import gurobipy as gp
import numpy as np
import pandas as pd
from gurobipy import GRB

from stile import (ARANCIO, GRIGIO, ROSSO, TEAL, VERDE, intestazione, plt, salva_dat, salva_dati,
                   salva_figura)


def frazione(x):
    """un numero non intero come frazione con denominatore al più 12 (6/5, -3/5); altrimenti con due decimali"""
    if not np.isfinite(x):
        return "inf" if x > 0 else "-inf"
    q = Fraction(x).limit_denominator(12)
    return str(q) if abs(float(q) - x) < 1e-9 else f"{x:.2f}"

## 1. DATI

In [ ]:
prodotti = ["1", "2", "3"]
mesi = list(range(1, 4))

# domanda (unità/mese): crescente, con il picco nel mese 3
domanda = {
    ("1", 1): 110, ("1", 2): 140, ("1", 3): 200,
    ("2", 1): 60,  ("2", 2): 80,  ("2", 3): 120,
    ("3", 1): 40,  ("3", 2): 40,  ("3", 3): 70,
}
costo_prod = {"1": 10, "2": 15, "3": 20}          # €/unità
costo_giac = {"1": 1, "2": 6 / 5, "3": 6 / 5}      # €/unità/mese
ore_unit = {"1": 1, "2": 3 / 2, "3": 2}              # ore macchina per unità
capacita = {1: 400, 2: 400, 3: 400}  # ore/mese
scorta_iniziale = {"1": 20, "2": 20, "3": 0}    # r_i0: scorta a inizio orizzonte (unità)
scorta_finale = {"1": 10, "2": 0, "3": 20}      # r_im: scorta richiesta a fine orizzonte (unità)
ultimo = mesi[-1]                               # mese m
mesi_s = mesi[:-1]                              # mesi 1..m-1: solo qui la scorta s_it è una variabile

df = pd.DataFrame(
    [(i, t, domanda[i, t], costo_prod[i], costo_giac[i], ore_unit[i], scorta_iniziale[i], scorta_finale[i])
     for i in prodotti for t in mesi],
    columns=["prodotto", "mese", "domanda", "costo_prod", "costo_giac", "ore_unit", "scorta_iniziale",
             "scorta_finale"])
salva_dati(df, "produzione_domanda")
salva_dati(pd.DataFrame({"mese": mesi, "capacita_ore": [capacita[t] for t in mesi]}),
           "produzione_capacita")


def costruisci_lp():
    """LP base: minimo costo di produzione + giacenza, servizio obbligatorio."""
    m = gp.Model("produzione_scorte")
    m.Params.OutputFlag = 0
    x = m.addVars(prodotti, mesi, name="x")           # quantità prodotta
    s = m.addVars(prodotti, mesi_s, name="s")         # scorta a fine mese, mesi 1..m-1
    # bilancio delle scorte (variabili a sinistra, dati a destra): primo mese, mesi intermedi, ultimo mese
    m.addConstrs((x[i, 1] - s[i, 1] == domanda[i, 1] - scorta_iniziale[i]
                  for i in prodotti), name="bilancio_primo")
    m.addConstrs((x[i, t] + s[i, t - 1] - s[i, t] == domanda[i, t]
                  for i in prodotti for t in mesi[1:-1]), name="bilancio")
    m.addConstrs((x[i, ultimo] + s[i, ultimo - 1] == domanda[i, ultimo] + scorta_finale[i]
                  for i in prodotti), name="bilancio_ultimo")
    # capacità della risorsa condivisa
    v_cap = m.addConstrs(
        (gp.quicksum(ore_unit[i] * x[i, t] for i in prodotti) <= capacita[t]
         for t in mesi), name="capacita")
    m.setObjective(
        gp.quicksum(costo_prod[i] * x[i, t] for i in prodotti for t in mesi)
        + gp.quicksum(costo_giac[i] * s[i, t] for i in prodotti for t in mesi_s), GRB.MINIMIZE)
    return m, x, s, v_cap

## 2. LP BASE: soluzione e duali

In [ ]:
intestazione("LP base: costo minimo con servizio obbligatorio")
m, x, s, v_cap = costruisci_lp()
m.optimize()
assert m.Status == GRB.OPTIMAL
print(f"Costo totale ottimo: {frazione(m.ObjVal)} €")

def scorta(i, t):
    """scorta a fine mese t: la variabile s_it per t < m, il dato r_im per t = m"""
    return s[i, t].X if t < ultimo else scorta_finale[i]


piano = pd.DataFrame(
    [(i, t, x[i, t].X, scorta(i, t)) for i in prodotti for t in mesi],
    columns=["prodotto", "mese", "produzione", "scorta"])
salva_dati(piano, "produzione_piano_ottimo")
print("\nPiano di produzione (unità/mese):")
print(piano.pivot(index="prodotto", columns="mese", values="produzione").apply(lambda c: c.map(frazione)))
print(f"\nScorte a fine mese (mese {ultimo}: la scorta finale richiesta r_i{ultimo}, un dato):")
print(piano.pivot(index="prodotto", columns="mese", values="scorta").apply(lambda c: c.map(frazione)))

print("\nVincoli di capacità: utilizzo, prezzo ombra e range di validità")
righe_duali = []
for t in mesi:
    uso = sum(ore_unit[i] * x[i, t].X for i in prodotti)
    c = v_cap[t]
    righe_duali.append((t, uso, capacita[t], c.Pi, c.SARHSLow, c.SARHSUp))
    print(f"  mese {t}: uso {frazione(uso)} ore su {capacita[t]} | "
          f"prezzo ombra {frazione(c.Pi)} €/ora | valido per b_t in [{frazione(c.SARHSLow)}, {frazione(c.SARHSUp)}]")
duali = pd.DataFrame(righe_duali, columns=["mese", "ore_usate", "capacita", "prezzo_ombra",
                                           "rhs_min", "rhs_max"])
salva_dati(duali, "produzione_duali_capacita")

# verifica del prezzo ombra per perturbazione (mese col duale più negativo:
# nella convenzione Gurobi, per un vincolo <= in un problema di minimo Pi <= 0)
t_star = duali.loc[duali["prezzo_ombra"].idxmin(), "mese"]
pi_star = duali["prezzo_ombra"].min()
m2, x2, s2, v2 = costruisci_lp()
v2[t_star].RHS = capacita[t_star] + 1
m2.optimize()
print(f"\nVerifica: +1 ora nel mese {t_star} → il costo varia di {frazione(m2.ObjVal - m.ObjVal)} € "
      f"(= prezzo ombra {frazione(pi_star)})")

## 3. FIGURE (dati per pgfplots + anteprima matplotlib)

In [ ]:
salva_dat(pd.DataFrame({
    "mese": mesi,
    **{f"x{i}": [x[i, t].X for t in mesi] for i in prodotti},
    **{f"s{i}": [scorta(i, t) for t in mesi] for i in prodotti},       # mese m: il dato r_im
    **{f"d{i}": [domanda[i, t] for t in mesi] for i in prodotti},
    "domtot": [sum(domanda[i, t] for i in prodotti) for t in mesi],
}), "cap05_piano")

fig = plt.figure(figsize=(10.5, 8.4))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.42, wspace=0.28)
ax1 = fig.add_subplot(gs[0, :])           # 1. ore macchina a_i x_it impilate e capacità b_t
ax2 = fig.add_subplot(gs[1, 0])           # 2. produzione x_it e domanda d_it per prodotto
ax3 = fig.add_subplot(gs[1, 1])           # 3. scorte s_it
colori = {i: c for i, c in zip(prodotti, [TEAL, ROSSO, VERDE])}
sotto = np.zeros(len(mesi))
for i in prodotti:
    ore_i = np.array([ore_unit[i] * x[i, t].X for t in mesi])
    ax1.bar(mesi, ore_i, bottom=sotto, width=0.6, color=colori[i], label=f"prodotto {i}: ore macchina $a_{i} x_{{{i}t}}$")
    sotto += ore_i
for k, t in enumerate(mesi):
    ax1.hlines(capacita[t], t - 0.4, t + 0.4, colors=ROSSO, linestyles="--", lw=1.8,
               label="capacità $b_t$ (ore disponibili)" if k == 0 else None)
    if sotto[k] >= capacita[t] - 1e-6:
        ax1.text(t, capacita[t] + 8, "satura", ha="center", fontsize=8, color=ROSSO)
ax1.plot([], [], " ", label="satura: vincolo di capacità attivo (tutte le ore usate)")      # voce di sola legenda
ax1.set_ylim(0, max(capacita.values()) * 1.12)
ax1.set_xticks(mesi); ax1.set_xlabel("mese"); ax1.set_ylabel("ore")
ax1.set_title("Ore macchina usate per prodotto e capacità (l'altezza della barra è il totale del mese)")
ax1.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1.0))   # fuori a destra: non copre le barre
larg = 0.26
for k, i in enumerate(prodotti):
    ax2.bar([t + (k - 1) * larg for t in mesi], [x[i, t].X for t in mesi], width=larg, color=colori[i],
            label=f"prodotto {i}: produzione $x_{{{i}t}}$")
for k, i in enumerate(prodotti):
    ax2.plot([t + (k - 1) * larg for t in mesi], [domanda[i, t] for t in mesi], "--", marker="D", ms=4,
             color=colori[i], mfc="white", lw=1.2, label=f"prodotto {i}: domanda $d_{{{i}t}}$")
ax2.set_ylim(0, max(domanda.values()) * 1.6)
ax2.set_xticks(mesi); ax2.set_xlabel("mese"); ax2.set_ylabel("unità")
ax2.set_title("Produzione e domanda per prodotto")
ax2.legend(fontsize=7, ncol=2, loc="upper left")
for i in prodotti:                        # scorte s_it (mesi 1..m-1) e, vuoto, il dato r_im del mese m
    ax3.plot(mesi_s, [s[i, t].X for t in mesi_s], marker="o", color=colori[i], label=f"prodotto {i}")
    ax3.plot([mesi_s[-1], ultimo], [s[i, mesi_s[-1]].X, scorta_finale[i]], "--", color=colori[i], lw=1)
    ax3.plot([ultimo], [scorta_finale[i]], "o", color=colori[i], mfc="white")
ax3.plot([], [], "o", color=GRIGIO, mfc="white", label=f"mese {ultimo}: scorta finale $r_{{i{ultimo}}}$ (dato)")
ax3.set_xticks(mesi); ax3.set_xlabel("mese"); ax3.set_ylabel("unità a scorta")
ax3.set_title("Scorte di fine mese")
ax3.set_ylim(-3, max([s[i, t].X for i in prodotti for t in mesi_s] + [10]) * 1.6)
ax3.legend(fontsize=8, loc="upper left")
salva_figura(fig, "cap05_piano_scorte")

# sensitività: costo ottimo al variare della capacità uniforme
intestazione("Sensitività: costo ottimo al variare della capacità")
percenti = np.arange(90, 136)
costi = []
for p in percenti:
    ms, xs_, ss_, vs = costruisci_lp()
    for t in mesi:
        vs[t].RHS = capacita[t] * p / 100
    ms.optimize()
    costi.append(ms.ObjVal if ms.Status == GRB.OPTIMAL else np.nan)
    if p % 5 == 0:
        print(f"  capacità al {p}%: " + (f"costo {frazione(costi[-1])} €" if ms.Status == GRB.OPTIMAL else "inammissibile"))

tab_cap = pd.DataFrame({"percento": percenti, "costo": costi}).dropna()
salva_dat(tab_cap, "cap05_capacita")

fig, ax = plt.subplots()
ax.plot(percenti, costi, "-o", ms=3, color=TEAL)
ax.axvline(100, color=ARANCIO, linestyle="--", label="capacità attuale")
ax.set_xlabel("capacità disponibile (% dell'attuale)")
ax.set_ylabel("costo totale ottimo (€)")
ax.set_title("Curva valore della capacità: convessa e decrescente")
ax.legend()
salva_figura(fig, "cap05_valore_capacita")

print("\nFatto: capitolo 5.")

---

Notebook generato da `python/lab05_produzione.py` con `python3 python/genera_notebook.py`:
le modifiche si fanno sullo script, non qui.

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testi, figure e dati [CC BY 4.0](https://github.com/fabiofurini/modellazione-pl/blob/main/LICENSE),
codice [MIT](https://github.com/fabiofurini/modellazione-pl/blob/main/LICENSE-CODE).